# Unit Tests — Silver & Gold Transformation Logic

Tests every business rule applied during the Silver transformation using a local Spark session. Each helper function mirrors the exact PySpark logic used in the Silver notebook so tests are self-contained and runnable without a running pipeline.

| Test Class | What is tested |
|------------|---------------|
| Deduplication | Window-based dedup keeps latest record per key |
| Text Standardization | initcap, trim, lowercase applied correctly |
| Order Time Slot | Hour → Breakfast / Lunch / Snacks / Dinner / Late Night |
| Delivery SLA Status | Delivery mins → Within / Near / Breached, boundary values |
| Price Tier | avg_cost_for_two → Budget / Mid-Range / Premium / Luxury |
| Rating Tier | rating → Excellent / Very Good / Good / Average / Below Average |
| Customer Segment | order count → Power User / Regular / Occasional / New |
| Amount Clamping | Negative totals replaced with 0 |
| Order Flags | is_delivered, is_cancelled, has_discount derived correctly |
| Referential Integrity | Unmatched FKs filtered out by inner join |
| Null-Safe Coalesce | NULL replaced with defined defaults |

In [ ]:
from pyspark.sql import Row, SparkSession
from pyspark.sql.functions import (
    col, coalesce, hour, initcap, lit, lower,
    row_number, to_timestamp, trim, when,
)
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("zomato-transformation-tests")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

PASS = "\u2713 PASS"
FAIL = "\u2717 FAIL"
results = []

def check(name, condition):
    status = PASS if condition else FAIL
    results.append((name, status))
    print(f"{status}  {name}")

print("Spark session ready.")

## Transformation Helper Functions

Each function mirrors the exact logic in the Silver notebook.

In [ ]:
def deduplicate(df, key_col, order_col):
    w = Window.partitionBy(key_col).orderBy(col(order_col).desc())
    return df.withColumn("_rn", row_number().over(w)).filter(col("_rn") == 1).drop("_rn")

def apply_time_slot(df, ts_col="order_placed_at"):
    h = hour(to_timestamp(col(ts_col)))
    return df.withColumn(
        "order_time_slot",
        when((h >= 6) & (h < 11), "Breakfast")
        .when((h >= 11) & (h < 15), "Lunch")
        .when((h >= 15) & (h < 18), "Snacks")
        .when((h >= 18) & (h < 23), "Dinner")
        .otherwise("Late Night")
    )

def apply_delivery_sla(df, mins_col="delivery_time_mins"):
    return df.withColumn(
        "delivery_sla_status",
        when(col(mins_col) <= 30, "Within SLA")
        .when(col(mins_col) <= 45, "Near SLA")
        .otherwise("SLA Breached")
    )

def apply_price_tier(df, cost_col="avg_cost_for_two"):
    return df.withColumn(
        "price_tier",
        when(col(cost_col) < 300, "Budget")
        .when(col(cost_col) < 700, "Mid-Range")
        .when(col(cost_col) < 1500, "Premium")
        .otherwise("Luxury")
    )

def apply_rating_tier(df, rating_col="rating"):
    return df.withColumn(
        "rating_tier",
        when(col(rating_col) >= 4.5, "Excellent")
        .when(col(rating_col) >= 4.0, "Very Good")
        .when(col(rating_col) >= 3.5, "Good")
        .when(col(rating_col) >= 3.0, "Average")
        .otherwise("Below Average")
    )

def apply_customer_segment(df, orders_col="total_orders_lifetime"):
    return df.withColumn(
        "customer_segment",
        when(col(orders_col) >= 50, "Power User")
        .when(col(orders_col) >= 15, "Regular")
        .when(col(orders_col) >= 5, "Occasional")
        .otherwise("New")
    )

def clamp_negative(df, amount_col="total_amount"):
    return df.withColumn(amount_col, when(col(amount_col) < 0, lit(0.0)).otherwise(col(amount_col)))

def apply_order_flags(df):
    return (
        df.withColumn("is_delivered", col("order_status") == "Delivered")
          .withColumn("is_cancelled", col("order_status") == "Cancelled")
          .withColumn("has_discount",  col("discount_amount") > 0)
    )

print("Helper functions defined.")

## Deduplication Tests

In [ ]:
df_dup   = spark.createDataFrame([Row(id="A", ts="2024-01-02", value=10), Row(id="A", ts="2024-01-01", value=5), Row(id="B", ts="2024-01-01", value=20)])
df_nodup = spark.createDataFrame([Row(id="A", ts="2024-01-01"), Row(id="B", ts="2024-01-02")])
df_3dup  = spark.createDataFrame([Row(id="X", ts="2024-03-01"), Row(id="X", ts="2024-01-01"), Row(id="X", ts="2024-02-01")])

r_dup    = deduplicate(df_dup, "id", "ts")
r_nodup  = deduplicate(df_nodup, "id", "ts")
r_3dup   = deduplicate(df_3dup, "id", "ts")

check("Dedup — removes duplicate keys",          r_dup.count() == 2)
check("Dedup — keeps latest record (value=10)",  r_dup.filter(col('id') == 'A').collect()[0]['value'] == 10)
check("Dedup — no-dup dataset unchanged",        r_nodup.count() == 2)
check("Dedup — _rn column removed after dedup",  "_rn" not in r_dup.columns)
check("Dedup — 3 dupes → keeps latest",          r_3dup.count() == 1 and r_3dup.collect()[0]['ts'] == '2024-03-01')

## Text Standardization Tests

In [ ]:
df_text = spark.createDataFrame([
    Row(name="john doe", email="USER@EXAMPLE.COM", city="mUmBAI", padded="  hello  ")
])
result = (
    df_text
    .withColumn("name",   initcap(trim(col("name"))))
    .withColumn("email",  lower(trim(col("email"))))
    .withColumn("city",   initcap(col("city")))
    .withColumn("padded", trim(col("padded")))
).collect()[0]

check("Text — initcap on name",             result['name']   == 'John Doe')
check("Text — lowercase email",             result['email']  == 'user@example.com')
check("Text — initcap on city",             result['city']   == 'Mumbai')
check("Text — trim removes whitespace",     result['padded'] == 'hello')

## Order Time Slot Tests

In [ ]:
ts_rows = [
    ("2024-06-15 07:30:00", "Breakfast"),
    ("2024-06-15 12:00:00", "Lunch"),
    ("2024-06-15 16:00:00", "Snacks"),
    ("2024-06-15 20:00:00", "Dinner"),
    ("2024-06-15 01:00:00", "Late Night"),
    ("2024-06-15 06:00:00", "Breakfast"),   # boundary
    ("2024-06-15 22:59:00", "Dinner"),      # boundary
]
df_ts = apply_time_slot(spark.createDataFrame([Row(order_placed_at=ts) for ts, _ in ts_rows]))
slots = [r['order_time_slot'] for r in df_ts.collect()]

for i, (ts, expected) in enumerate(ts_rows):
    check(f"Time Slot — {ts} → {expected}", slots[i] == expected)

check("Time Slot — all 5 slots producible",
      {"Breakfast", "Lunch", "Snacks", "Dinner", "Late Night"} == set(slots))

## Delivery SLA Status Tests

In [ ]:
sla_cases = [(25, "Within SLA"), (38, "Near SLA"), (55, "SLA Breached"), (30, "Within SLA"), (45, "Near SLA")]
df_sla    = apply_delivery_sla(spark.createDataFrame([Row(delivery_time_mins=m) for m, _ in sla_cases]))
sla_vals  = [r['delivery_sla_status'] for r in df_sla.collect()]

for i, (mins, expected) in enumerate(sla_cases):
    check(f"SLA — {mins} min → {expected}", sla_vals[i] == expected)

## Price Tier Tests

In [ ]:
tier_cases = [(200, "Budget"), (500, "Mid-Range"), (1000, "Premium"), (2500, "Luxury")]
df_tier    = apply_price_tier(spark.createDataFrame([Row(avg_cost_for_two=c) for c, _ in tier_cases]))
tier_vals  = [r['price_tier'] for r in df_tier.collect()]

for i, (cost, expected) in enumerate(tier_cases):
    check(f"Price Tier — {cost} → {expected}", tier_vals[i] == expected)

check("Price Tier — all 4 tiers producible", set(tier_vals) == {"Budget", "Mid-Range", "Premium", "Luxury"})

## Rating Tier Tests

In [ ]:
rating_cases = [(4.8, "Excellent"), (4.2, "Very Good"), (3.7, "Good"), (3.1, "Average"), (2.5, "Below Average"),
                (4.5, "Excellent"), (4.0, "Very Good")]  # boundaries
df_rating    = apply_rating_tier(spark.createDataFrame([Row(rating=r) for r, _ in rating_cases]))
rating_vals  = [r['rating_tier'] for r in df_rating.collect()]

for i, (rating, expected) in enumerate(rating_cases):
    check(f"Rating Tier — {rating} → {expected}", rating_vals[i] == expected)

## Customer Segment Tests

In [ ]:
seg_cases = [(100, "Power User"), (20, "Regular"), (8, "Occasional"), (2, "New"),
             (0, "New"), (50, "Power User"), (15, "Regular")]  # boundaries
df_seg    = apply_customer_segment(spark.createDataFrame([Row(total_orders_lifetime=o) for o, _ in seg_cases]))
seg_vals  = [r['customer_segment'] for r in df_seg.collect()]

for i, (orders, expected) in enumerate(seg_cases):
    check(f"Customer Segment — {orders} orders → {expected}", seg_vals[i] == expected)

## Amount Clamping Tests

In [ ]:
clamp_cases = [(-50.0, 0.0), (250.0, 250.0), (0.0, 0.0), (-9999.99, 0.0)]
df_clamp    = clamp_negative(spark.createDataFrame([Row(total_amount=a) for a, _ in clamp_cases]))
clamp_vals  = [r['total_amount'] for r in df_clamp.collect()]

for i, (inp, expected) in enumerate(clamp_cases):
    check(f"Amount Clamping — {inp} → {expected}", clamp_vals[i] == expected)

## Order Flag Derivation Tests

In [ ]:
df_flags = apply_order_flags(spark.createDataFrame([
    Row(order_status="Delivered",  discount_amount=50.0),
    Row(order_status="Cancelled",  discount_amount=0.0),
    Row(order_status="Placed",     discount_amount=0.0),
]))
rows = df_flags.collect()

check("Order Flags — Delivered → is_delivered=True",  rows[0]['is_delivered'] is True)
check("Order Flags — Delivered → is_cancelled=False", rows[0]['is_cancelled'] is False)
check("Order Flags — discount>0 → has_discount=True", rows[0]['has_discount'] is True)
check("Order Flags — Cancelled → is_cancelled=True",  rows[1]['is_cancelled'] is True)
check("Order Flags — Cancelled → is_delivered=False", rows[1]['is_delivered'] is False)
check("Order Flags — no discount → has_discount=False",rows[1]['has_discount'] is False)

## Referential Integrity Tests

In [ ]:
orders_df    = spark.createDataFrame([Row(order_id="O1", customer_id="C1"), Row(order_id="O2", customer_id="C99")])
customers_df = spark.createDataFrame([Row(customer_id="C1")])
valid_orders = orders_df.join(customers_df, "customer_id", "inner")

orders_df2   = spark.createDataFrame([Row(order_id="O1", customer_id="C1"), Row(order_id="O2", customer_id="C2")])
customers_df2= spark.createDataFrame([Row(customer_id="C1"), Row(customer_id="C2")])
valid_all    = orders_df2.join(customers_df2, "customer_id", "inner")

dlv_df = spark.createDataFrame([Row(delivery_id="D1", order_id="O1"), Row(delivery_id="D2", order_id="O99")])
ord_df = spark.createDataFrame([Row(order_id="O1")])
valid_dlv = dlv_df.join(ord_df, "order_id", "inner")

check("Ref Integrity — unmatched orders filtered out",  valid_orders.count() == 1)
check("Ref Integrity — matched orders all preserved",   valid_all.count() == 2)
check("Ref Integrity — deliveries filtered by orders",  valid_dlv.count() == 1)

## Null-Safe Coalesce Tests

In [ ]:
df_null = spark.createDataFrame([Row(tip=None), Row(tip=25.0)])
result  = df_null.withColumn("tip", coalesce(col("tip"), lit(0.0))).collect()

check("Coalesce — NULL → default 0.0",    result[0]['tip'] == 0.0)
check("Coalesce — non-null preserved",    result[1]['tip'] == 25.0)

df_bool = spark.createDataFrame([Row(flag=None)])
result2 = df_bool.withColumn("flag", coalesce(col("flag"), lit(False))).collect()
check("Coalesce — NULL bool → False",     result2[0]['flag'] is False)

## Summary

In [ ]:
passed = sum(1 for _, s in results if s == PASS)
failed = sum(1 for _, s in results if s == FAIL)

print(f"\n{'='*55}")
print(f"  Transformation Logic Test Results")
print(f"{'='*55}")
print(f"  Total  : {len(results)}")
print(f"  Passed : {passed}")
print(f"  Failed : {failed}")
print(f"  Result : {'ALL TESTS PASSED' if failed == 0 else f'{failed} TEST(S) FAILED'}")
print(f"{'='*55}")

spark.stop()

if failed > 0:
    for name, status in results:
        if status == FAIL:
            print(f"  {FAIL}  {name}")
    raise AssertionError(f"{failed} transformation test(s) failed.")